智能工具筛选。当工具太多时，用子模型筛选最相关的几个工具。

参数：
- model ：用于工具筛选的子模型
- max_tools ：限定可以调用的工具总数
- always_include ：指定的工具不被计数

提供必要的大模型

In [1]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(override=True)

main_model = init_chat_model(
    model="qwen3.7-flash",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
)

tool_model = init_chat_model(
    model="qwen3.7-plus",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
)

In [9]:
from langchain.agents import create_agent
from langchain.agents.middleware import LLMToolSelectorMiddleware
from langchain.messages import HumanMessage
from langchain.tools import tool

@tool(name_or_callable="get_weather_tool")
def get_weather(city: str) -> str:
    """查询指定城市天气"""
    return f"{city}今天天气晴朗"

@tool(name_or_callable="get_news_tool")
def get_news() -> str:
    """查询今日国内的新闻概要"""
    return "今天国内的新闻概要：中方三艘邮轮过霍尔木兹海峡"

@tool(name_or_callable="calculate_tool")
def calculate(num1: int, num2: int) -> int:
    """
    执行数学计算

    Args:
        num1: 第一个加数
        num2: 第二个加数

    Returns:
        两数之和
    """
    return num1 + num2

@tool(name_or_callable="search_stock_tool")
def search_stock(symbol: str) -> str:
    """
    查询股票行情

    Args:
        symbol: 股票代码

    Returns:
        股票行情
    """
    return "该股票今天行情不错"

agent = create_agent(
    model=main_model,
    tools=[get_weather, get_news, calculate, search_stock],
    middleware=[
        LLMToolSelectorMiddleware(
            model=tool_model,
            max_tools=1,     #还有一个tool
            always_include=["get_weather_tool"]  # 必需选择的tool
        )
    ],
)

response = agent.invoke({
    "messages": HumanMessage("北京今天天气如何？今日国内的新闻概要"),
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

北京今天天气如何？今日国内的新闻概要
================================== Ai Message ==================================
Tool Calls:
  get_weather_tool (call_e0e7216db9104508b92c5b42)
 Call ID: call_e0e7216db9104508b92c5b42
  Args:
    city: 北京
  get_news_tool (call_b414e17ee2c14b158c890cda)
 Call ID: call_b414e17ee2c14b158c890cda
  Args:
================================= Tool Message =================================
Name: get_weather_tool

北京今天天气晴朗
================================= Tool Message =================================
Name: get_news_tool

今天国内的新闻概要：中方三艘邮轮过霍尔木兹海峡
================================== Ai Message ==================================

北京今天的天气为晴朗。今日国内新闻方面，有中方三艘邮轮通过霍尔木兹海峡的消息。
